# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema_content = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()
print(schema_content["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [8]:
feature_vector = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions)   AS total_impressions,
            SUM(gsc_clicks)        AS total_clicks,
            AVG(gsc_avg_position)  AS avg_position,
            SUM(ga4_sessions)      AS total_sessions,
            SUM(scroll_events)     AS total_scroll_events
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        d.content_hash_id,
        d.total_impressions,
        d.total_clicks,
        d.avg_position,
        d.total_sessions,
        d.total_scroll_events,
        c.word_count,
        c.content_type,
        c.main_intent,
        c.competition_level,
        c.search_volume,
        c.cpc,
        c.backlinks,
        c.content_created_date,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
    FROM daily_agg d
    JOIN {TABLES['dim_content']} c ON d.content_hash_id = c.content_hash_id
""").df()

print("Shape:", feature_vector.shape)
feature_vector.head()

Shape: (176738, 15)


,content_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events,word_count,content_type,main_intent,competition_level,search_volume,cpc,backlinks,content_created_date,content_age_days
0,content_0263d5f9b7a2ecd4,1.0,0.0,9.000000,NaN,NaN,3246,keyword article,informational,LOW,0,0.0,0,2025-10-07,175
1,content_04c67f3541177192,331.0,2.0,14.129210,NaN,NaN,3168,keyword article,informational,LOW,0,0.0,0,2025-10-07,175
2,content_05acc92c165f4386,33.0,0.0,9.225529,NaN,NaN,4135,keyword article,commercial,LOW,20,0.0,9,2025-10-07,175
3,content_0f30e04e709c7b5d,145.0,0.0,8.470926,NaN,NaN,3211,keyword article,informational,LOW,0,0.0,0,2025-10-07,175
4,content_1207efddce873942,461.0,0.0,14.859827,NaN,NaN,3465,keyword article,informational,LOW,0,0.0,0,2025-10-07,175


In [9]:
print("Missing values per column:")
print(feature_vector.isna().sum())

Missing values per column:
content_hash_id             0
total_impressions           0
total_clicks                0
avg_position                0
total_sessions          48726
total_scroll_events     48726
word_count              55315
content_type                0
main_intent             15839
competition_level       16786
search_volume           15932
cpc                     15932
backlinks               70163
content_created_date        0
content_age_days            0
dtype: int64


In [10]:
# Numeric signals where missing likely means "no activity/data" -> fill with 0
zero_fill_cols = ["total_sessions", "total_scroll_events", "backlinks"]
feature_vector[zero_fill_cols] = feature_vector[zero_fill_cols].fillna(0)

# Numeric signals where missing likely means "unknown value", not zero -> fill with median
median_fill_cols = ["word_count", "search_volume", "cpc"]
for col in median_fill_cols:
    feature_vector[col] = feature_vector[col].fillna(feature_vector[col].median())

# Categorical -> fill with an explicit "unknown" category, not a guess
categorical_cols = ["main_intent", "competition_level"]
for col in categorical_cols:
    feature_vector[col] = feature_vector[col].fillna("unknown")

print("Remaining missing values:")
print(feature_vector.isna().sum())

Remaining missing values:
content_hash_id         0
total_impressions       0
total_clicks            0
avg_position            0
total_sessions          0
total_scroll_events     0
word_count              0
content_type            0
main_intent             0
competition_level       0
search_volume           0
cpc                     0
backlinks               0
content_created_date    0
content_age_days        0
dtype: int64


Feature vector built from:

    -fact_content_daily_performance (March 2026, gsc_data_available IS TRUE), aggregated per content item: total_impressions, total_clicks, avg_position, total_sessions, total_scroll_events

    -dim_content, joined on content_hash_id: word_count, content_type, main_intent, competition_level, search_volume, cpc, backlinks, content_age_days (derived from content_created_date)

Final shape: 176,738 rows × 15 columns (after joining daily aggregates to content metadata).

Missing value handling:

    -total_sessions, total_scroll_events, backlinks → filled with 0 (missing plausibly means "no activity recorded," not "unknown")

    -word_count, search_volume, cpc → filled with the median (missing here means "unknown value," not zero — a page with 0 words isn't realistic)

    -main_intent, competition_level → filled with an explicit "unknown" category, rather than guessing a value, so the model can see the gap instead of a fabricated default

Categorical handling: content_type, main_intent, competition_level are categorical and will need encoding (e.g. one-hot) before any model training — not done yet in this notebook, noted for Week 5.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

total_impressions, total_clicks, avg_position, total_sessions, total_scroll_events

Meaning: aggregated GSC/GA4 signals within the March 2026 window.
Missing: total_sessions/total_scroll_events filled with 0 (no GA4 tracking ≠ no traffic, but treated as 0 for simplicity — noted as a limitation).
Available before decision point? Yes — all summed from report_dates within the observation window itself, none extend past it.


word_count

Meaning: current word count of the content item, from dim_content.
Missing: filled with median.
Available before decision point? Risky. dim_content is a dimension table describing the current state of the content — if a page was edited after March 2026 (word count changed), this column could reflect a future revision, not the word count as it was during my observation window. I don't have a "word count as of March 31" snapshot, only "word count now." This is a leakage risk worth flagging, not silently trusting.


content_type, main_intent, competition_level

Meaning: categorical metadata about the content and its target keyword.
Missing: filled with "unknown" category.
Available before decision point? Yes, most likely — these describe the content/keyword's inherent classification, not something that changes based on the outcome I'm predicting. Low leakage risk.


search_volume, cpc

Meaning: external SEO keyword metrics (search demand, cost-per-click) for the content's target keyword.
Missing: filled with median.
Available before decision point? Yes — these describe the keyword itself, independent of how the page performed in March; they don't derive from the outcome.


backlinks

Meaning: count of backlinks pointing to the content.
Missing: filled with 0.
Available before decision point? Risky, same issue as word_count. dim_content gives me the current backlink count, not a snapshot as of the end of March — backlinks accumulate over time, so this may include links acquired after my observation window.

content_age_days

Meaning: days between content_created_date and 2026-03-31 (end of window).
Missing: none (computed).
Available before decision point? Yes — creation date is fixed in the past, and I anchored the calculation to the end of my window, not to "today."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
update_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_content,
        COUNT(*) FILTER (WHERE content_updated_date > DATE '2026-03-31') AS updated_after_window,
        ROUND(100.0 * COUNT(*) FILTER (WHERE content_updated_date > DATE '2026-03-31') / COUNT(*), 1) AS pct_updated_after
    FROM {TABLES['dim_content']}
    WHERE content_hash_id IN (SELECT content_hash_id FROM feature_vector)
""").df()
update_check

,total_content,updated_after_window,pct_updated_after
0,176738,148782,84.2


In [15]:
optimize_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_content,
        COUNT(*) FILTER (WHERE last_optimized_date IS NOT NULL) AS has_optimization_record,
        COUNT(*) FILTER (WHERE last_optimized_date > DATE '2026-03-31') AS optimized_after_window
    FROM {TABLES['dim_content']}
    WHERE content_hash_id IN (SELECT content_hash_id FROM feature_vector_view)
""").df()
optimize_check

,total_content,has_optimization_record,optimized_after_window
0,176738,39764,39764


Leakage hunt findings:

I tested my suspicion from Section 2 about dim_content reflecting current state, not state as of the observation window (2026-03-31).

Test 1 — content_updated_date: 84.2% of content items in my feature vector (148,782 of 176,738) were updated after March 31, 2026. This confirms that word_count and backlinks — both pulled from dim_content — very likely reflect the content's current state, not its state during my March observation window. Using them as features for a March-based label is a real leakage risk: the model could be learning from information that didn't exist yet at decision time.

Test 2 — last_optimized_date: Only 39,764 rows (22.5%) have any optimization record at all, and every single one of them was optimized after the March window closed. This field is essentially unusable as a March-time feature — it's either empty or describes something that happened after my window. I'm treating it as excluded, not just risky.

What I'm doing about it: I'm keeping word_count and backlinks in this notebook's feature vector for demonstration purposes, but flagging both as leakage-risk features that should NOT be trusted for real modeling without a proper point-in-time snapshot (e.g. a content-history table, if one exists, or restricting to content items that were never updated after the window). For Week 5 modeling, I would drop them or find a time-safe version.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields, and why:

word_count, backlinks — Not fully excluded from this notebook's demo vector, but flagged as leakage-risk and would be excluded from real modeling. Test 1 in Section 3 showed 84.2% of content items were updated after the March window closed, meaning these dim_content fields likely reflect the content's current state, not its state during the observation window. Using them risks the model learning from information that didn't exist yet at decision time.

last_optimized_date, optimization_eligible_date — Excluded entirely. Test 2 showed every optimization record in my slice occurred after the March window, and 77.5% of rows have no record at all. This field is tied to FlyRank's own optimization workflow (a product decision), not an observed search/content signal — exactly the kind of product-context field the data-use rules say to keep out of modeling.
is_published, is_deleted — Excluded as features. These are status flags useful for filtering the dataset (e.g. only include published, non-deleted content) but carry no meaningful signal about visibility or movement on their own.

keyword_hash_id, url_hash_id — Excluded as features, kept only as potential join/grouping keys. Per the data contract rules, hashed IDs are pseudonyms for grouping and deduplication only — they carry no real information and should never be treated as model inputs.

provider_used, model_used — Excluded. These likely describe which internal tool or AI provider generated/assisted the content — an internal production detail, not an observed search or content-performance signal, and out of scope for Lane 1's question.

sessions_paid, sessions_social, sessions_referral, sessions_direct, AI-referral columns (sessions_ai, ai_chatgpt, etc.) — Excluded, same reasoning as Week 3's data contract: paid/social traffic is a different dynamic than organic search, and AI-referral sessions are too sparse (per the lane guide's density table) to include without adding noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.